In [2]:
import os
import sys
import time

import pandas as pd
import numpy as np
import anndata as ad

import tacco as tc
import scanpy as sc
from collections import Counter
import gc
import matplotlib.pyplot as plt

import pickle
import re

from collections import defaultdict
import itertools

from easySCFpy import loadH5

In [1]:
sample_list_st = [
    "P1","P2","P3","P4","P5","P6","P7","P8","P9","P10","P11","P12","P13","P14","P15","P16","P26","P27","P30","P32","P33","P34","P35","P36","P37","P38","P39",
    "P40_1","P40_2","P42","P43_1","P43_2","P44","P45","P46","P47","P48","P49"
]

In [ ]:
mye_sub_celltype = pd.read_csv(f"E:/Projects/BRCA/results/ST/Macrophage/mye_sub_celltype.csv", index_col=0)

In [ ]:
# TACCO
for spID in sample_list_st[0:len(sample_list_st)]:
    print("processing sample: " + spID)
    object = sc.read_h5ad(f"E:/Projects/BRCA/Intermediate_Data/ST/object_unified/{spID}_seg.h5ad")
    
    # Load and add cell types
    celltype_info = pd.read_csv(f"E:/Projects/BRCA/results/ST/Niche2607/{spID}/celltype_{spID}.csv")
    celltype_info.set_index('bin', inplace=True)
    object.obs['celltype'] = celltype_info.reindex(object.obs.index)
    
    # Add myeloid cell subtype information
    object.obs['sub_celltype'] = object.obs['celltype'].astype(str)
    object.obs.loc[object.obs['sub_celltype'].isin(['Macrophages','DC','Mast']), 'sub_celltype'] = 'undefined'
    sp_mye_info = mye_sub_celltype[mye_sub_celltype['sample'] == spID]
    sp_mye_info = sp_mye_info[sp_mye_info.index.isin(object.obs.index.tolist())]
    object.obs.loc[sp_mye_info.index.tolist(),'sub_celltype'] = sp_mye_info['sub_celltype']
    object.obs.loc[object.obs['sub_celltype'].isin(['cDC','pDC']),'celltype'] = 'DC'
    object.obs.loc[object.obs['sub_celltype'] == 'Mast','celltype'] = 'Mast'
    object.obs.loc[object.obs['sub_celltype'].isin(['Macro_C1','Macro_C2','Macro_C3','Macro_C4']),'celltype'] = 'Macrophages'
    object = object[~(object.obs['sub_celltype'].isin(['delete','undefined','unknown']))]
    object = object[~(object.obs['celltype'].isin(['delete','undefined','unknown']))]
    # Add region information
    region_info = pd.read_csv(f"E:/Projects/BRCA/results/ST/cluster_ljy/all_sample_region/{spID}_region.csv", index_col=0)
    region_info.loc[region_info['region'] == "others", 'region'] = 'undefined'
    object.obs['region'] = region_info['region'].reindex(object.obs.index)
    object.obs['region'] = object.obs['region'].fillna('undefined')
    # Remove epithelial cells without assigned regions
    object = object[~((object.obs['region'].isin(["undefined", "Fibroadenoma"])) & (object.obs['celltype'] == "Epithelial"))]
    # Add labels for colocalization analysis
    object.obs['cooccur_celltype'] = object.obs['celltype'].astype(str)
    object.obs['cooccur_sub_celltype'] = object.obs['sub_celltype'].astype(str)
    object.obs.loc[object.obs['cooccur_celltype'] == 'Epithelial', 'cooccur_celltype'] = object.obs.loc[object.obs['cooccur_celltype'] == 'Epithelial', 'region'] + "_Epithelial"
    object.obs.loc[object.obs['cooccur_sub_celltype'] == 'Epithelial', 'cooccur_sub_celltype'] = object.obs.loc[object.obs['cooccur_sub_celltype'] == 'Epithelial', 'region'] + "_Epithelial"
    object = object[~(object.obs['celltype'] == 'Unknown')]
    object = object[~(object.obs['cooccur_sub_celltype'] == 'undefined')]
    print("cooccur_celltype:")
    print(Counter(object.obs['cooccur_celltype']))
    print("cooccur_sub_celltype:")
    print(Counter(object.obs['cooccur_sub_celltype']))
    
    # Run tacco
    delta_sp = object.uns['spatial']['slice1']['scalefactors']['spot_diameter_fullres'] / 2 * 5
    max_dis_sp = object.uns['spatial']['slice1']['scalefactors']['spot_diameter_fullres'] / 2 * 310
    print(f"delta used is: {delta_sp}")
    print(f"max distance used is: {max_dis_sp}")
    # Run by celltype
    tc.tl.co_occurrence(object, annotation_key='cooccur_celltype', center_key='cooccur_celltype', result_key='type-type', position_key='spatial', delta_distance=delta_sp, max_distance=max_dis_sp, sparse=True)
    
    if not os.path.exists(f"E:/Projects/BRCA/results/ST/tacco/{spID}"):
        os.makedirs(f"E:/Projects/BRCA/results/ST/tacco/{spID}")
    with open(f"E:/Projects/BRCA/results/ST/tacco/{spID}/tacco_res_celltype_{spID}.pkl", "wb") as f:
        pickle.dump(object.uns['type-type'], f)
    
    # Run by cell subtype
    tc.tl.co_occurrence(object, annotation_key='cooccur_sub_celltype', center_key='cooccur_sub_celltype', result_key='subtype-subtype', position_key='spatial', delta_distance=delta_sp, max_distance=max_dis_sp, sparse=True)
    with open(f"E:/Projects/BRCA/results/ST/tacco/{spID}/tacco_res_sub_celltype_{spID}.pkl", "wb") as f:
        pickle.dump(object.uns['subtype-subtype'], f)
    
    object.uns.pop('type-type')
    object.uns.pop('subtype-subtype')
    # Save h5ad
    print('saving object to h5ad ...')
    sc.write(f'E:/Projects/BRCA/Intermediate_Data/ST/object_unified/{spID}_seg.h5ad', object)

In [ ]:
# Summarize enrichment of all major cell types in each major epithelial region
for epi_region in ['IDC', 'DCIS', 'ADH', 'IP', 'EPC']:
    for spID in sample_list_st:
        celltype_info = pd.read_csv(f"E:/Projects/BRCA/results/ST/Niche2607/{spID}/celltype_{spID}.csv")
        celltype_info.set_index('bin', inplace=True)
    
        # Add region information
        region_info = pd.read_csv(f"E:/Projects/BRCA/results/ST/cluster_ljy/all_sample_region/{spID}_region.csv", index_col=0)
        region_info.loc[region_info['region'] == "others", 'region'] = 'undefined'
    
        epi_info_df = celltype_info[celltype_info['celltype'] == "Epithelial"]
        epi_info_df['region'] = region_info['region'].reindex(epi_info_df.index)
    
        # Check whether any group in this sample has fewer than 1,000 cells
        if Counter(epi_info_df['region'] == epi_region)[True] < 1000:
            continue
    
        with open(f"E:/Projects/BRCA/results/ST/tacco/{spID}/tacco_res_celltype_{spID}.pkl", "rb") as f:
            sp_cooc = pickle.load(f)
        
        # Compare the following cell types
        celltype_list = ['Adipocytes', 'B', 'Basal', 'DC', 'Endothelial', 'Fibroblasts', 'Macrophages', 'Mast', 'Pericytes', 'Plasma', 'T']
        for celltype in celltype_list:
            if Counter(celltype_info['celltype'])[celltype] < 400:
                continue
            center_list = sp_cooc['center'].tolist()
            anchor_index = center_list.index(epi_region + '_Epithelial')
            observe_index = center_list.index(celltype)
            cooccur_df = sp_cooc['occ'][anchor_index,observe_index,0:60]
            cooccur_df = pd.DataFrame(cooccur_df, columns = [spID])
        
            if os.path.exists(f"E:/Projects/BRCA/results/ST/tacco/celltype_cooccur/{epi_region}_{celltype}_cooccur.csv"):
                cooccur_df_all = pd.read_csv(f"E:/Projects/BRCA/results/ST/tacco/celltype_cooccur/{epi_region}_{celltype}_cooccur.csv", index_col = 0)
                cooccur_df_all = pd.concat([cooccur_df_all, cooccur_df], axis = 1)
                cooccur_df_all.to_csv(f"E:/Projects/BRCA/results/ST/tacco/celltype_cooccur/{epi_region}_{celltype}_cooccur.csv")
            else:
                cooccur_df.to_csv(f"E:/Projects/BRCA/results/ST/tacco/celltype_cooccur/{epi_region}_{celltype}_cooccur.csv")

In [ ]:
# Summarize enrichment of all cell subtypes in each major epithelial region
for epi_region in ['IDC', 'DCIS', 'ADH', 'IP', 'EPC']:
    for spID in sample_list_st:
        object = sc.read_h5ad(f"E:/Projects/BRCA/Intermediate_Data/ST/object_unified/{spID}_seg.h5ad")
        #sub_celltype_info = object.obs['sub_celltype']
    
        # Add region information
        region_info = pd.read_csv(f"E:/Projects/BRCA/results/ST/cluster_ljy/all_sample_region/{spID}_region.csv", index_col=0)
        region_info.loc[region_info['region'] == "others", 'region'] = 'undefined'
    
        epi_info_df = object.obs[object.obs['sub_celltype'] == "Epithelial"]
        epi_info_df['region'] = region_info['region'].reindex(epi_info_df.index)
    
        # Check whether any group in this sample has fewer than 1,000 cells
        if Counter(epi_info_df['region'] == epi_region)[True] < 1000:
            continue
    
        with open(f"E:/Projects/BRCA/results/ST/tacco/{spID}/tacco_res_sub_celltype_{spID}.pkl", "rb") as f:
            sp_cooc = pickle.load(f)
        
        sub_celltype_list = ['cDC','pDC','Macro_C1','Macro_C2','Macro_C3','Macro_C4','Mast']
        
        for celltype in sub_celltype_list:
            if Counter(object.obs['sub_celltype'])[celltype] < 200:
                continue
            center_list = sp_cooc['center'].tolist()
            anchor_index = center_list.index(epi_region + '_Epithelial')
            observe_index = center_list.index(celltype)
            cooccur_df = sp_cooc['occ'][anchor_index,observe_index,0:60]
            cooccur_df = pd.DataFrame(cooccur_df, columns = [spID])
        
            if os.path.exists(f"E:/Projects/BRCA/results/ST/tacco/sub_celltype_cooccur/{epi_region}_{celltype}_cooccur.csv"):
                cooccur_df_all = pd.read_csv(f"E:/Projects/BRCA/results/ST/tacco/sub_celltype_cooccur/{epi_region}_{celltype}_cooccur.csv", index_col = 0)
                cooccur_df_all = pd.concat([cooccur_df_all, cooccur_df], axis = 1)
                cooccur_df_all.to_csv(f"E:/Projects/BRCA/results/ST/tacco/sub_celltype_cooccur/{epi_region}_{celltype}_cooccur.csv")
            else:
                cooccur_df.to_csv(f"E:/Projects/BRCA/results/ST/tacco/sub_celltype_cooccur/{epi_region}_{celltype}_cooccur.csv")